# Baseline Evaluation — `whisper-small-khmer-v2`

**Project:** Intelligent Khmer Voice-to-Text Telegram Bot with LLM Post-Processing
**Notebook purpose:** Establish the starting-point accuracy of the existing fine-tuned
checkpoint (`seanghay/whisper-small-khmer-v2`) *before* any further fine-tuning or
data changes. This is the number everything else in the project gets compared against.

## What this notebook does
1. Mounts Google Drive for persistent result storage (Colab's local disk is wiped on disconnect)
2. Loads the baseline checkpoint from Hugging Face
3. Runs inference on the held-out FLEURS Khmer test split (`data/eval/`)
4. Computes CER (primary metric) and WER (secondary) using `src/utils/metrics.py`
5. Saves numeric results + example transcriptions (good and bad) to `docs/baseline_results.md`

## Requirements
- `transformers`, `peft`, `librosa`, `jiwer`, `soundfile`
- T4 GPU runtime (Runtime → Change runtime type → T4 GPU)

## Related task
`[DATA-02] Load & Run Baseline Model`, `[EVAL-02] Run Baseline Evaluation`
(see Trello board)

In [ ]:
%pip install transformers peft librosa jiwer soundfile

#### Mount the drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#### Create the folder for project inside drive

In [ ]:
import os
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/khmer-stt-bot'
os.makedirs(f'{DRIVE_PROJECT_DIR}/results', exist_ok=True)
os.makedirs(f'{DRIVE_PROJECT_DIR}/checkpoints', exist_ok=True)
print("Drive mounted. Project folder ready at:", DRIVE_PROJECT_DIR)

## Step 2: Load & Run Baseline Model

Load the existing fine-tuned checkpoint (`seanghay/whisper-small-khmer-v2`) and confirm it
runs correctly on a few sample files before running the full evaluation. No training or
fine-tuning happens here — just a sanity check that the model loads, runs on GPU, and
produces Khmer text output in the format we expect (16kHz input → text output).

#### Load model:

In [ ]:
import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor

MODEL_ID = "seanghay/whisper-small-khmer-v2"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

processor = WhisperProcessor.from_pretrained(MODEL_ID)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID).to(device)
model.eval()

print("Model loaded:", MODEL_ID)

#### Set config for model to focus on transcribe with only khmer language.

In [ ]:
# Bake the "Khmer, transcribe" instruction directly into the generation config
# as forced_decoder_ids, bypassing the language=/task= lookup path entirely —
# that path assumes a lang_to_id mapping this older checkpoint doesn't have.
model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="km", task="transcribe"
)

#### Make the fuction by use the model to transcribe the data (voice).

In [ ]:
import librosa

prompt_length = len(model.generation_config.forced_decoder_ids)
MAX_TOTAL_TOKENS = 448
SAFETY_MARGIN = 10

def transcribe(audio_path, sample_rate=16000):
    audio, sr = librosa.load(audio_path, sr=sample_rate, mono=True)
    duration_sec = len(audio) / sample_rate

    dynamic_max_tokens = min(
        int(duration_sec * 40) + 80,
        MAX_TOTAL_TOKENS - prompt_length - SAFETY_MARGIN
    )

    inputs = processor(audio, sampling_rate=sample_rate, return_tensors="pt").to(device)

    with torch.no_grad():
        predicted_ids = model.generate(
            inputs["input_features"],
            max_new_tokens=dynamic_max_tokens,
        )

    text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    return text

### Testing the model with few sample.

In [ ]:
import time
import glob

sample_files = sorted(glob.glob("data/eval/fleurs_km/*.wav"))[:3]

for f in sample_files:
    start = time.time()
    text = transcribe(f)
    elapsed = time.time() - start
    print(f"File: {f}")
    print(f"  Transcription: {text}")
    print(f"  Time: {elapsed:.2f}s")
    print()

### Testing the new evaluate.

In [ ]:
result = transcribe("/content/drive/MyDrive/khmer-stt-bot/data/eval/real_001.ogg")
print(result)

## Step 3: Download FLEURS Khmer Test Split

Pulling the Khmer subset of Google FLEURS from Hugging Face — this becomes our held-out
evaluation set (`data/eval/`). Using the official `test` split so our baseline CER/WER is
directly comparable to the numbers reported for `whisper-small-khmer-v2`.

In [ ]:
!pip install -q datasets

In [ ]:
from datasets import load_dataset

# "km_kh" = Khmer, Cambodia — FLEURS' language code for Khmer
fleurs_km = load_dataset("google/fleurs", "km_kh", split="test")
print(fleurs_km)
print("Number of test samples:", len(fleurs_km))

#### Save the data (voice) with the ground truth (the text of that voice) together

In [ ]:
import os
import soundfile as sf

EVAL_DIR = "data/eval/fleurs_km"
os.makedirs(EVAL_DIR, exist_ok=True)

transcript_lines = []

for i, sample in enumerate(fleurs_km):
    audio_array = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]
    text = sample["transcription"]

    filename = f"fleurs_{i:04d}.wav"
    filepath = os.path.join(EVAL_DIR, filename)
    sf.write(filepath, audio_array, sr)

    transcript_lines.append(f"{filename}\t{text}")

# Save ground-truth transcripts alongside the audio
with open(os.path.join(EVAL_DIR, "transcripts.tsv"), "w", encoding="utf-8") as f:
    f.write("\n".join(transcript_lines))

print(f"Saved {len(fleurs_km)} audio files + transcripts to {EVAL_DIR}")

In [ ]:
!pip install jiwer

## Step 4: Run Full Baseline Evaluation

Run `whisper-small-khmer-v2` on the entire FLEURS Khmer test set and compute CER/WER
against ground-truth transcripts. This establishes the baseline number for the whole
project — every later improvement (fine-tuning, LLM correction) gets compared against
this.

#### install the scoring library and load ground truth:

In [ ]:
!pip install -q jiwer


In [ ]:
import csv

EVAL_DIR = "data/eval/fleurs_km"

# Load the filename -> correct transcript mapping we saved earlier
ground_truth = {}
with open(f"{EVAL_DIR}/transcripts.tsv", encoding="utf-8") as f:
    reader = csv.reader(f, delimiter="\t")
    for row in reader:
        filename, text = row[0], row[1]
        ground_truth[filename] = text

print(f"Loaded {len(ground_truth)} ground-truth transcripts")

#### transcribe every file and collect predictions:

In [ ]:
import os

predictions = {}

for filename in sorted(ground_truth.keys()):
    filepath = os.path.join(EVAL_DIR, filename)
    predictions[filename] = transcribe(filepath)

print(f"Transcribed {len(predictions)} files")

In [ ]:
import json
import os

RESULTS_PATH = f"{DRIVE_PROJECT_DIR}/results/baseline_predictions.json"

# Load existing progress if this is a resume, otherwise start fresh
if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH, encoding="utf-8") as f:
        predictions = json.load(f)
    print(f"Resuming — {len(predictions)} files already done")
else:
    predictions = {}

remaining = [f for f in sorted(ground_truth.keys()) if f not in predictions]
print(f"{len(remaining)} files left to transcribe")

for i, filename in enumerate(remaining):
    filepath = os.path.join(EVAL_DIR, filename)
    predictions[filename] = transcribe(filepath)

    # Save every 20 files — cheap insurance against losing progress
    if i % 20 == 0:
        with open(RESULTS_PATH, "w", encoding="utf-8") as f:
            json.dump(predictions, f, ensure_ascii=False, indent=2)
        print(f"  progress: {len(predictions)}/{len(ground_truth)} saved")

# Final save
with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

print(f"Done. {len(predictions)} predictions saved to {RESULTS_PATH}")

#### compute CER and WER:

In [ ]:
from jiwer import cer, wer

refs = [ground_truth[f] for f in sorted(ground_truth.keys())]
hyps = [predictions[f] for f in sorted(ground_truth.keys())]

baseline_cer = cer(refs, hyps)
baseline_wer = wer(refs, hyps)

print(f"Baseline CER: {baseline_cer:.4f} ({baseline_cer*100:.2f}%)")
print(f"Baseline WER: {baseline_wer:.4f} ({baseline_wer*100:.2f}%)")

In [ ]:
# Check what's still alive in memory after reconnect
print("predictions exists:", 'predictions' in dir())
print("ground_truth exists:", 'ground_truth' in dir())

if 'predictions' in dir():
    print("Number of predictions:", len(predictions))

In [ ]:
for filename in sorted(predictions.keys())[:3]:
    ref = ground_truth[filename]
    hyp = predictions[filename]
    print(f"File: {filename}")
    print(f"  Reference : {ref}")
    print(f"  Prediction: {hyp}")
    print(f"  Ref word count (by space): {len(ref.split())}")
    print(f"  Hyp word count (by space): {len(hyp.split())}")
    print()

In [ ]:
import json

os.makedirs(f"{DRIVE_PROJECT_DIR}/results", exist_ok=True)

with open(f"{DRIVE_PROJECT_DIR}/results/baseline_predictions.json", "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

with open(f"{DRIVE_PROJECT_DIR}/results/baseline_ground_truth.json", "w", encoding="utf-8") as f:
    json.dump(ground_truth, f, ensure_ascii=False, indent=2)

print("Saved to Drive.")